Import Packages

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [6]:
!pip install statsbombpy
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [statsbombpy]


In [7]:
import numpy as np
import pandas as pd
from statsbombpy import sb
import pandas as pd
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [8]:
# Import matches and events
pairs = [(72, 30), (72, 107), (53, 106), (53, 315), (49, 3), (49, 107)] # WWC_2019, WWC_2023, W_Eruo_2022, W_Euro_2025, NWSL_2018, NWSL_2023

matches = pd.concat(
    [sb.matches(competition_id=c, season_id=s) for c, s in pairs],
    ignore_index=True,
)

events = [sb.events(m) for m in matches.match_id]


In [9]:
# Combine all event DataFrames into a single DataFrame
events_df = pd.concat(events, ignore_index=True)

# Extract shots from events
shots = events_df[events_df["type"] == "Shot"].copy()

# Remove penalties
shots = shots[shots["shot_type"] != "Penalty"].copy()

# Encode target
shots["goal"] = (shots["shot_outcome"] == "Goal").astype(int)


Stats bomb measures the field as 120x80, therefore subtracting 120 will determine the `x` distance from the goal. We are subtracting 40 from `y` since the goal sits in the center of the y axis and we want the distance from the goal.

In [10]:
def calc_angle_dist(X):
    x, y = X["x"], X["y"]
    distance = np.hypot(120 - x, 40 - y)
    a_x, a_y = 120 - x, 36 - y
    b_x, b_y = 120 - x, 44 - y
    cos = (a_x*b_x + a_y*b_y) / (np.hypot(a_x, a_y) * np.hypot(b_x, b_y))
    angle = np.arccos(np.clip(cos, -1, 1))
    return pd.DataFrame({"distance": distance, "angle": angle}, index=X.index)

def clean_flags(X):
    return X.fillna(False).astype(int)

In [11]:
shots["x"] = shots["location"].str[0]
shots["y"] = shots["location"].str[1]

In [12]:
sorted(shots['shot_outcome'].unique())

['Blocked',
 'Goal',
 'Off T',
 'Post',
 'Saved',
 'Saved Off Target',
 'Saved to Post',
 'Wayward']

In [13]:
# Calculate distance and angle
shots[["distance", "angle"]] = calc_angle_dist(shots)

flags = ["under_pressure", "shot_one_on_one", "shot_follows_dribble", "shot_first_time"]

preprocess = ColumnTransformer([
    ("body", OneHotEncoder(drop=["Right Foot"], sparse_output=False, handle_unknown="ignore"), ["shot_body_part"]), # Right Foot is Baseline
    ("type", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), ["shot_type"]), # Corner is Baseline
    ("num", StandardScaler(), ["x", "y", "distance", "angle"]),
    ("flags", FunctionTransformer(clean_flags, feature_names_out="one-to-one"), flags),
], remainder="passthrough")

In [14]:
# Define columns in DF
cols = ["x", "y", "distance", "angle", "shot_body_part", "shot_type",
        "under_pressure", "shot_one_on_one", "shot_follows_dribble", "shot_first_time"]
preprocess.set_output(transform="pandas")
X_prep = preprocess.fit_transform(shots[cols])
X_prep.columns

Index(['body__shot_body_part_Head', 'body__shot_body_part_Left Foot',
       'body__shot_body_part_Other', 'type__shot_type_Free Kick',
       'type__shot_type_Kick Off', 'type__shot_type_Open Play', 'num__x',
       'num__y', 'num__distance', 'num__angle', 'flags__under_pressure',
       'flags__shot_one_on_one', 'flags__shot_follows_dribble',
       'flags__shot_first_time'],
      dtype='object')

In [15]:
pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

In [16]:
len(X_prep)

9146

In [17]:
# Re-split the data now that 'x', 'y', 'distance', and 'angle' columns have been added to 'shots'
# Slice the input features strictly to the columns we intend to preprocess to avoid passing unhandled metadata IDs
X_train, X_test, y_train, y_test = train_test_split(shots[cols], shots["goal"], test_size=0.2, random_state=42)

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9131147540983606


Random Forest

In [18]:
RF_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(random_state=42))
])


# Prepend parameters with 'model__' to route them to the RandomForestClassifier step in the Pipeline
params = {
    "model__n_estimators": [1, 50, 100, 200, 300],
    "model__max_depth": [5, 10, 15, 100],
    "model__max_features": [3, 5, 7, 10]
}

grid = GridSearchCV(RF_pipeline, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)



KeyboardInterrupt: 

In [ ]:
RF_pipeline_updated = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(max_depth=5, max_features=3, n_estimators=200,random_state=42))
])

RF_pipeline_updated.fit(X_train, y_train)

# Predict
y_pred_rf = RF_pipeline_updated.predict(X_test)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred_rf))

In [ ]:
KNN_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", KNeighborsClassifier())
])


# Prepend parameters with 'model__' to route them to the RandomForestClassifier step in the Pipeline
params = {
    "model__n_neighbors": np.arange(1, 1000, 2)
}

grid = GridSearchCV(KNN_pipeline, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
